In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import joblib

In [2]:
sms = pd.read_csv('https://drive.google.com/uc?id=1d_KwNxymj9aNaS8LfUWLSRy1qwZYifsF')

In [3]:
sms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Kategori  1143 non-null   object
 1   Pesan     1143 non-null   object
dtypes: object(2)
memory usage: 18.0+ KB


In [4]:
df = pd.read_csv("hf://datasets/ashu0311/SMS_Spam_Multilingual_Collection_Dataset/data-augmented.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
df = df[['text_id', 'labels']]

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text_id  5572 non-null   object
 1   labels   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [7]:
sms2 = pd.read_csv('https://drive.google.com/uc?id=1oYpYXhyQSmyD9knESpXcTuJWvx3KvAzL', encoding='latin-1')

In [8]:
sms2.head(2)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN


In [9]:
sms2 = sms2[['v1', 'v2']]

In [10]:
sms2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   v1      5572 non-null   object
 1   v2      5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [11]:
sms['Kategori'].value_counts()

,count
Kategori,
spam,574
ham,569


In [12]:
sms2['v1'].value_counts()

,count
v1,
ham,4825
spam,747


In [13]:
df['labels'].value_counts()

,count
labels,
ham,4825
spam,747


In [14]:
sms.duplicated().sum()

np.int64(1)

In [15]:
sms.drop_duplicates(inplace=True)

In [16]:
sms2.duplicated().sum()

np.int64(403)

In [17]:
sms2.drop_duplicates(inplace=True)

In [18]:
df.duplicated().sum()

np.int64(424)

In [19]:
df.drop_duplicates(inplace=True)

In [20]:
sms2.isna().sum()

,0
v1,0
v2,0


In [21]:
df.isna().sum()

,0
text_id,0
labels,0


In [22]:
sms.isna().sum()

,0
Kategori,0
Pesan,0


In [23]:
df_clean = df[['text_id', 'labels']].rename(columns={'text_id': 'text', 'labels': 'label'})

sms2_clean = sms2[['v2', 'v1']].rename(columns={'v2': 'text', 'v1': 'label'})

sms_clean = sms[['Pesan', 'Kategori']].rename(columns={'Pesan': 'text', 'Kategori': 'label'})

In [24]:
data = pd.concat([df_clean, sms2_clean, sms_clean], ignore_index=True)

data = data.dropna(subset=['text', 'label'])
data = data.drop_duplicates()

In [25]:
mapping = {
    'ham': 'not phishing',
    'spam': 'phishing'
}

data['label'] = data['label'].map(mapping)

In [29]:
data

,text,label
0,"Pergi sampai jurong point, gila.. Tersedia han...",not phishing
1,"Awalnya, ia bergurau dengan Wif U...",not phishing
2,Free entry in 2 a wkly comp to win FA Cup fina...,phishing
3,U dun berkata begitu awal hor... U c sudah kem...,not phishing
4,"Saya tidak berpikir dia pergi ke usf, dia ting...",not phishing
...,...,...
11454,Yg ragu sm bulet/datar atau yg pgn ikutan deba...,not phishing
11455,"Semangat yang ibu gita, ibu putri dan bapak ad...",not phishing
11456,"nama1, minta database kamu sama view dan contr...",not phishing
11457,Dapatkan GRATIS 1 cappuccino (hot/ice) & Freza...,phishing


In [27]:
data.to_csv('data_spam_full.csv', index=False)

In [28]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import joblib

class SentenceTransformerWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='paraphrase-multilingual-MiniLM-L12-v2'):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return self.model.encode(X, show_progress_bar=False)

spam_pipeline = Pipeline([
    ('embedder', SentenceTransformerWrapper('paraphrase-multilingual-MiniLM-L12-v2')),
    ('classifier', LogisticRegression(max_iter=1000))
])

X_text = data['text'].tolist()
y_label = data['label'].tolist()

X_train_text, X_test_text, y_train_label, y_test_label = train_test_split(
    X_text, y_label, test_size=0.2, random_state=42
)

print("Training")
spam_pipeline.fit(X_train_text, y_train_label)

y_pred_pipe = spam_pipeline.predict(X_test_text)
print("Akurasi Pipeline:", accuracy_score(y_test_label, y_pred_pipe))
print(classification_report(y_test_label, y_pred_pipe))


print("'spam_pipeline.pkl'")
joblib.dump(spam_pipeline, 'spam_pipeline.pkl')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Training
Akurasi Pipeline: 0.9833916083916084
              precision    recall  f1-score   support

not phishing       0.99      0.99      0.99      1950
    phishing       0.94      0.94      0.94       338

    accuracy                           0.98      2288
   macro avg       0.97      0.97      0.97      2288
weighted avg       0.98      0.98      0.98      2288

'spam_pipeline.pkl'


['spam_pipeline.pkl']